# G2: Predicted-Analysis Generation

This notebook is the G2 generation entry point. It accepts a reflection document and segment analysis produced by `2_segment_inference.ipynb`, then calls the shared assessment-and-feedback runner. Run inference for the same document first and save its predicted-analysis artifact under approved protected storage. Never substitute G3 human labels.

Run from the `prebi_v1` project root with the `.venv` kernel. The current sample document source is the provisional test split; use it only for an approved pilot or final test, not for prompt or model tuning. Never print, commit, or send protected reflection data or predictions to an unapproved provider or tracing workspace.

## 1. Detect Current Notebook File Path

Jupyter does not reliably expose the active notebook path to Python. Run from the project root and set the notebook paths explicitly.

In [ ]:
from pathlib import Path

WORKSPACE_DIR = Path.cwd()
G1_G3_NOTEBOOK_PATH = WORKSPACE_DIR / "G1_G3_Demo.ipynb"
G2_NOTEBOOK_PATH = WORKSPACE_DIR / "G2_Demo.ipynb"
assert WORKSPACE_DIR.name == "prebi_v1", "Run this notebook from the prebi_v1 project root."
assert G2_NOTEBOOK_PATH.exists(), "G2_Demo.ipynb was not found in the project root."

## 2. Rename Notebook to `G1_G3_Demo.ipynb`

This step is idempotent: it renames a known legacy G1/G3 filename only when the target name is absent. It never renames the active G2 notebook.

In [ ]:
legacy_names = ("2_direct_generation.ipynb", "2_g1_g3_generation_demo.ipynb")
if not G1_G3_NOTEBOOK_PATH.exists():
    legacy_path = next(
        (WORKSPACE_DIR / name for name in legacy_names if (WORKSPACE_DIR / name).exists()),
        None,
    )
    if legacy_path is None:
        raise FileNotFoundError("No known G1/G3 notebook filename was found.")
    legacy_path.rename(G1_G3_NOTEBOOK_PATH)
print(f"G1/G3 notebook ready: {G1_G3_NOTEBOOK_PATH.name}")

## 3. Create New Notebook File `G2_Demo.ipynb`

Create the requested notebook only if it is missing. When this setup is run from the existing G2 notebook, read and preserve the current file instead of replacing it.

In [ ]:
import nbformat

created_new_g2_notebook = not G2_NOTEBOOK_PATH.exists()
if created_new_g2_notebook:
    g2_notebook = nbformat.v4.new_notebook()
    nbformat.write(g2_notebook, G2_NOTEBOOK_PATH)
else:
    g2_notebook = nbformat.read(G2_NOTEBOOK_PATH, as_version=4)
print("Created G2 notebook." if created_new_g2_notebook else "Existing G2 notebook preserved.")

## 4. Add Starter Cells to `G2_Demo.ipynb`

If creating a fresh G2 notebook, seed it with a title and the shared generation imports. Existing notebook content is never overwritten.

In [ ]:
g2_starter_cells = [
    nbformat.v4.new_markdown_cell(
        "## G2 generation\n\nLoad the predicted scope and component analysis, then call the shared G2 generation runner."
    ),
    nbformat.v4.new_code_cell(
        "from reflection_assessment_feedback import (\n"
        "    IntermediateAnalysis, RubricGenerationRunner, make_generation_request,\n"
        ")\n\n"
        "# TODO: load the predicted analysis artifact from the approved inference pipeline."
    ),
]
if created_new_g2_notebook:
    g2_notebook.cells.extend(g2_starter_cells)
    nbformat.write(g2_notebook, G2_NOTEBOOK_PATH)
else:
    print("G2 starter scaffold is already maintained in this notebook.")

## 5. Validate Files in Workspace

List the notebooks in the project root and assert that both experiment workbooks exist.

In [ ]:
notebook_paths = sorted(WORKSPACE_DIR.glob("*.ipynb"))
print([path.name for path in notebook_paths])
assert G1_G3_NOTEBOOK_PATH.is_file()
assert G2_NOTEBOOK_PATH.is_file()

## G2 Generation Workflow

### Load reflection text without gold labels

This cell explicitly reads only `document_id`, `segment_id`, `text`, and `segment_order`. It deduplicates candidate copies of identical segments only after verifying that their text and order agree; it does not load annotation targets.

In [ ]:
import os

import pandas as pd
from reflection_assessment_feedback import (
    EXPERT_RUBRIC,
    ReflectionDocument,
    ReflectionSegment,
)

SPLIT_PATH = WORKSPACE_DIR / "data" / "provisional_segment_splits" / "provisional_test_modeling_wide.csv"
reflection_rows = pd.read_csv(
    SPLIT_PATH,
    usecols=["document_id", "segment_id", "text", "segment_order"],
)
reflection_rows["document_id"] = reflection_rows["document_id"].astype(str)
reflection_rows["segment_id"] = reflection_rows["segment_id"].astype(str)
reflection_rows["text"] = reflection_rows["text"].fillna("").astype(str)
reflection_rows["segment_order"] = pd.to_numeric(
    reflection_rows["segment_order"], errors="raise"
).astype(int)
document_ids = sorted(reflection_rows["document_id"].dropna().unique())
DOCUMENT_INDEX = 0
if not 0 <= DOCUMENT_INDEX < len(document_ids):
    raise IndexError("DOCUMENT_INDEX is outside the available document list.")
selected_document_id = document_ids[DOCUMENT_INDEX]
document_rows = reflection_rows.loc[
    reflection_rows["document_id"] == selected_document_id
].copy()
consistency = document_rows.groupby("segment_id").agg(
    text_variants=("text", "nunique"),
    order_variants=("segment_order", "nunique"),
)
if (consistency > 1).any().any():
    raise ValueError("Candidate copies disagree on segment text or source order.")
document_rows = document_rows.drop_duplicates("segment_id").sort_values("segment_order")
document = ReflectionDocument(
    document_id=selected_document_id,
    segments=[
        ReflectionSegment(
            segment_id=row.segment_id,
            text=row.text,
            order=row.segment_order,
        )
        for row in document_rows.itertuples(index=False)
    ],
)
print(f"Selected document contains {len(document.segments)} ordered segments.")

### Load predicted analysis

Run `2_segment_inference.ipynb` with the same `DOCUMENT_INDEX` as this notebook, then set its inference and write switches to `True` once all four direct-baseline checkpoints are complete. It writes one document-scoped JSON file to protected `PREBI_DATA_ROOT/g2/predicted-analysis.json` in the shape `{"document_id": "...", "analysis": {"source": "predicted", "segments": [...]}}`. Missing predictions leave this cell in a waiting state; G3 human annotations must never be substituted.

In [ ]:
import json

from reflection_assessment_feedback import IntermediateAnalysis

predicted_analysis = None
protected_root = os.getenv("PREBI_DATA_ROOT")
if protected_root:
    prediction_path = Path(protected_root) / "g2" / "predicted-analysis.json"
    if prediction_path.is_file():
        prediction_package = json.loads(prediction_path.read_text(encoding="utf-8"))
        if prediction_package.get("document_id") != document.document_id:
            raise ValueError("Prediction artifact does not match the selected document.")
        predicted_analysis = IntermediateAnalysis.model_validate(
            prediction_package["analysis"]
        )
        if predicted_analysis.source != "predicted":
            raise ValueError("G2 requires predicted, not human, segment analysis.")
        print(f"Predicted analysis loaded for {len(predicted_analysis.segments)} segments.")
    else:
        print("Waiting for the G2 inference pipeline to produce its prediction artifact.")
else:
    print("Set PREBI_DATA_ROOT to the approved protected data directory when predictions are ready.")

### Generate G2 assessment and feedback

Provider processing and LangSmith tracing are separate opt-ins. Run only with approved document data, model-provider processing, and tracing retention. To enable tracing, configure `LANGSMITH_API_KEY` and `LANGSMITH_PROJECT` in the project `.env`, then set `ENABLE_LANGSMITH_TRACING = True`; the Python generation span and nested LangChain model call will be sent to that project. Traces can contain the full reflection, prompt, and model output. Generated runs remain in memory.

In [ ]:
import importlib
import os

from dotenv import dotenv_values, load_dotenv

langsmith_dotenv_path = WORKSPACE_DIR / ".env"
langsmith_dotenv = dotenv_values(langsmith_dotenv_path)
load_dotenv(langsmith_dotenv_path, override=False)
conflicting_langsmith_settings = [
    name
    for name in ("LANGSMITH_API_KEY", "LANGSMITH_PROJECT", "LANGSMITH_ENDPOINT")
    if langsmith_dotenv.get(name)
    and os.getenv(name)
    and langsmith_dotenv[name] != os.getenv(name)
]
if conflicting_langsmith_settings:
    raise RuntimeError(
        "LangSmith settings differ between .env and the active kernel environment: "
        f"{', '.join(conflicting_langsmith_settings)}. The kernel environment takes "
        "precedence; update or unset the conflicting value and restart the kernel. "
        "Secret values are not displayed."
    )

from langchain_google_genai import ChatGoogleGenerativeAI
import reflection_assessment_feedback.generator as generation_module
from reflection_assessment_feedback import make_generation_request

generation_module = importlib.reload(generation_module)
RubricGenerationRunner = generation_module.RubricGenerationRunner

APPROVE_EXTERNAL_PROCESSING = False
ENABLE_LANGSMITH_TRACING = False
REPETITIONS = 2
assert APPROVE_EXTERNAL_PROCESSING, "Confirm provider processing is approved before running."
assert predicted_analysis is not None, "G2 inference predictions are not available yet."
assert os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"), (
    "Configure GOOGLE_API_KEY or GEMINI_API_KEY in the approved environment."
)
if ENABLE_LANGSMITH_TRACING:
    assert os.getenv("LANGSMITH_API_KEY"), (
        "Configure LANGSMITH_API_KEY in the local .env or approved environment."
    )
    assert os.getenv("LANGSMITH_PROJECT"), (
        "Configure LANGSMITH_PROJECT in the local .env or approved environment."
    )
    assert os.getenv("LANGSMITH_ENDPOINT", "https://eu.api.smith.langchain.com").rstrip("/") == (
        "https://eu.api.smith.langchain.com"
    ), "Tracing is restricted to the LangSmith EU endpoint."

chat_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
runner = RubricGenerationRunner(
    chat_model,
    enable_langsmith_tracing=ENABLE_LANGSMITH_TRACING,
)
g2_request = make_generation_request(
    condition="G2",
    document=document,
    rubric=EXPERT_RUBRIC,
    analysis=predicted_analysis,
)
g2_runs = [
    runner.generate(g2_request, repetition=index + 1)
    for index in range(REPETITIONS)
]
print(f"Completed {len(g2_runs)} G2 generations.")
if ENABLE_LANGSMITH_TRACING:
    print(
        "LangSmith tracing targeted EU endpoint "
        f"for project {os.environ['LANGSMITH_PROJECT']}; check warnings above for upload failures."
    )